# 09.2 — Auditoría de EVA curada

Compuerta posterior a la curación: valida unicidad del target, nulos, finitud, fórmula del rendimiento y cobertura territorial/temporal.

In [ ]:
from pathlib import Path
import subprocess, sys
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
REPO_REF = 'feature/SCRUM-15'
REPO_DIR = Path('/content/suelosabio') if IN_COLAB else Path.cwd()
if IN_COLAB:
    if not (REPO_DIR / '.git').exists():
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_REF, 'https://github.com/cybercolombia/suelosabio.git', str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', 'fetch', 'origin', REPO_REF], cwd=REPO_DIR, check=True)
        subprocess.run(['git', 'checkout', REPO_REF], cwd=REPO_DIR, check=True)
        subprocess.run(['git', 'pull', '--ff-only', 'origin', REPO_REF], cwd=REPO_DIR, check=True)
PIPELINE_DIR = REPO_DIR / 'notebooks' / 'ClimatePipeline'
if not PIPELINE_DIR.exists():
    PIPELINE_DIR = next((p for p in [Path.cwd(), Path.cwd() / 'ClimatePipeline', Path.cwd().parent / 'ClimatePipeline'] if (p / 'DatasetConfig.py').exists()), None)
if PIPELINE_DIR is None:
    raise FileNotFoundError('No se encontró ClimatePipeline.')
sys.path.insert(0, str(PIPELINE_DIR)) if str(PIPELINE_DIR) not in sys.path else None
from DatasetConfig import cargar_configuracion_datasets
from ClimateProcessingUtils import escribir_parquet_atomico, escribir_json_atomico, escribir_texto_atomico
from CropYieldProcessing import CURATION_VERSION, CURATED_AUDIT_VERSION, audit_curated_eva
import pandas as pd
from IPython.display import display
CONFIG = cargar_configuracion_datasets(in_colab=IN_COLAB)


In [ ]:
EJECUTAR_AUDITORIA_CURADA = False
SOBRESCRIBIR_RESULTADOS = False
INPUT_PATH = CONFIG.processed_root / 'agricultura_curada' / f'version={CURATION_VERSION}' / 'eva_curada.parquet'
OUTPUT_DIR = CONFIG.processed_root / 'auditorias_agricultura' / 'capa=eva_curada' / f'auditoria={CURATED_AUDIT_VERSION}'
print({'audit_version': CURATED_AUDIT_VERSION, 'entrada': str(INPUT_PATH), 'salida': str(OUTPUT_DIR), 'ejecutar': EJECUTAR_AUDITORIA_CURADA})
if not EJECUTAR_AUDITORIA_CURADA:
    print('Auditoría curada desactivada.')
else:
    eva_curada = pd.read_parquet(INPUT_PATH)
    auditoria = audit_curated_eva(eva_curada)
    for nombre, tabla in auditoria.items():
        escribir_parquet_atomico(tabla, OUTPUT_DIR / f'{nombre}.parquet', SOBRESCRIBIR_RESULTADOS)
    resumen = auditoria['summary'].iloc[0].to_dict()
    escribir_json_atomico({'curation_version': CURATION_VERSION, **resumen}, OUTPUT_DIR / 'manifest.json', SOBRESCRIBIR_RESULTADOS)
    reporte = f'# Auditoría EVA curada\n\n- Estado: `{resumen["estado"]}`\n- Filas: {resumen["filas"]:,}\n- Llaves duplicadas: {resumen["llaves_duplicadas"]:,}\n- Fórmulas inconsistentes: {resumen["filas_formula_inconsistente"]:,}\n'
    escribir_texto_atomico(reporte, OUTPUT_DIR / 'AuditoriaEvaCurada.md', SOBRESCRIBIR_RESULTADOS)
    display(auditoria['summary'])
